# 🐄 Cow Behavior Dataset Setup

This notebook automates the download and setup of the CBVD-5 Cow Behavior Video Dataset from Kaggle.

## Prerequisites

Before running this notebook, you need to:

1. **Create a Kaggle account** at https://www.kaggle.com
2. **Get your Kaggle API credentials**:
   - Go to your Kaggle profile → Account tab
   - Click "Create New API Token"
   - This downloads `kaggle.json` with your credentials
3. **Place the credentials file** in the correct location:
   - **Linux/Mac**: `~/.kaggle/kaggle.json`
   - **Windows**: `C:\Users\<username>\.kaggle\kaggle.json`
   - Or set the `KAGGLE_CONFIG_DIR` environment variable

## What this notebook does:

- ✅ Installs required packages
- ✅ Downloads the CBVD-5 dataset (~6GB)
- ✅ Extracts only the needed directories (`videos/` and `labelframes/`)
- ✅ Places them in the correct project structure
- ✅ Verifies the setup is complete
- ✅ Downloads YOLO pre-trained weights

**Total download size**: ~6GB (videos + labelframes only)

---

## 1. Install Required Packages

In [ ]:
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

# Install required packages
packages = ['kaggle', 'tqdm']

for package in packages:
    try:
        __import__(package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        install_package(package)
        print(f"✅ {package} installed successfully")

print("\n🎉 All packages ready!")

## 2. Import Libraries and Check Kaggle API

In [ ]:
import os
import zipfile
import shutil
import urllib.request
from pathlib import Path
from tqdm import tqdm

# Test Kaggle API
try:
    import kaggle
    from kaggle.api.kaggle_api_extended import KaggleApi
    
    # Initialize Kaggle API
    api = KaggleApi()
    api.authenticate()
    print("✅ Kaggle API authenticated successfully!")
    
    # Test API with a simple call
    user = api.get_config_value('username')
    print(f"👤 Logged in as: {user}")
    
except Exception as e:
    print(f"❌ Kaggle API error: {e}")
    print("\n🔧 Please ensure:")
    print("   1. You have a kaggle.json file with your API credentials")
    print("   2. The file is in ~/.kaggle/kaggle.json (Linux/Mac) or C:\\Users\\<username>\\.kaggle\\kaggle.json (Windows)")
    print("   3. The file permissions are correct (chmod 600 ~/.kaggle/kaggle.json on Linux/Mac)")
    raise

## 3. Set Up Directory Structure

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DOWNLOAD_DIR = PROJECT_ROOT / "temp_download"
VIDEOS_DIR = DATA_DIR / "videos"
LABELFRAMES_DIR = DATA_DIR / "labelframes"

# Create necessary directories
DOWNLOAD_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Download directory: {DOWNLOAD_DIR}")

# Check current data status
videos_exist = VIDEOS_DIR.exists() and any(VIDEOS_DIR.glob('*.mp4'))
labelframes_exist = LABELFRAMES_DIR.exists() and any(LABELFRAMES_DIR.rglob('*.jpg'))

print(f"\n📊 Current status:")
print(f"   Videos directory: {'✅ EXISTS' if videos_exist else '❌ MISSING'}")
print(f"   Labelframes directory: {'✅ EXISTS' if labelframes_exist else '❌ MISSING'}")

if videos_exist and labelframes_exist:
    print("\n🎉 Dataset already exists! You can skip the download steps.")
else:
    print("\n⬇️ Dataset download required.")

## 4. Download CBVD-5 Dataset from Kaggle

In [ ]:
# Dataset information
DATASET_NAME = "fandaoerji/cbvd-5cow-behavior-video-dataset"
DATASET_ZIP = "cbvd-5cow-behavior-video-dataset.zip"

def download_dataset():
    """Download the dataset from Kaggle"""
    print(f"📥 Downloading {DATASET_NAME}...")
    print(f"⚠️  This is a large dataset (~6GB), please be patient...")
    
    try:
        # Download to temporary directory
        api.dataset_download_files(
            dataset=DATASET_NAME,
            path=DOWNLOAD_DIR,
            quiet=False
        )
        
        zip_path = DOWNLOAD_DIR / DATASET_ZIP
        if zip_path.exists():
            print(f"✅ Download completed: {zip_path}")
            print(f"📊 File size: {zip_path.stat().st_size / (1024**3):.2f} GB")
            return zip_path
        else:
            raise FileNotFoundError(f"Downloaded file not found: {zip_path}")
            
    except Exception as e:
        print(f"❌ Download failed: {e}")
        raise

# Download if needed
if not (videos_exist and labelframes_exist):
    zip_path = download_dataset()
else:
    print("⏭️ Skipping download - dataset already exists")
    zip_path = None

## 5. Extract Dataset Files

In [ ]:
def extract_needed_directories(zip_path):
    """Extract only the videos and labelframes directories"""
    print(f"📦 Extracting files from {zip_path}...")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get list of all files in zip
        all_files = zip_ref.namelist()
        
        # Filter for videos and labelframes directories
        videos_files = [f for f in all_files if 'videos/' in f and f.endswith('.mp4')]
        labelframes_files = [f for f in all_files if 'labelframes/' in f and f.endswith('.jpg')]
        
        print(f"📺 Found {len(videos_files)} video files")
        print(f"🖼️ Found {len(labelframes_files)} labelframe images")
        
        # Create progress bars
        def extract_with_progress(files, desc):
            for file in tqdm(files, desc=desc):
                try:
                    # Extract to temporary location first
                    zip_ref.extract(file, DOWNLOAD_DIR)
                except Exception as e:
                    print(f"⚠️ Warning: Could not extract {file}: {e}")
        
        # Extract videos
        if videos_files and not videos_exist:
            extract_with_progress(videos_files, "Extracting videos")
        
        # Extract labelframes
        if labelframes_files and not labelframes_exist:
            extract_with_progress(labelframes_files, "Extracting labelframes")
    
    print("✅ Extraction completed!")

# Extract if we downloaded
if zip_path and not (videos_exist and labelframes_exist):
    extract_needed_directories(zip_path)
else:
    print("⏭️ Skipping extraction - dataset already exists")

## 6. Move Files to Correct Project Structure

In [ ]:
def move_to_project_structure():
    """Move extracted files to the correct project structure"""
    print("📂 Organizing files into project structure...")
    
    # Find extracted directories in download folder
    extracted_videos = None
    extracted_labelframes = None
    
    for root, dirs, files in os.walk(DOWNLOAD_DIR):
        if 'videos' in root and any(f.endswith('.mp4') for f in files):
            extracted_videos = root
        if 'labelframes' in root and any(f.endswith('.jpg') for f in files):
            extracted_labelframes = root
    
    # Move videos directory
    if extracted_videos and not videos_exist:
        print(f"📺 Moving videos from {extracted_videos} to {VIDEOS_DIR}")
        if VIDEOS_DIR.exists():
            shutil.rmtree(VIDEOS_DIR)
        shutil.move(extracted_videos, VIDEOS_DIR)
        print(f"✅ Videos moved successfully")
    
    # Move labelframes directory
    if extracted_labelframes and not labelframes_exist:
        print(f"🖼️ Moving labelframes from {extracted_labelframes} to {LABELFRAMES_DIR}")
        if LABELFRAMES_DIR.exists():
            shutil.rmtree(LABELFRAMES_DIR)
        shutil.move(extracted_labelframes, LABELFRAMES_DIR)
        print(f"✅ Labelframes moved successfully")

# Move files if we extracted them
if zip_path and not (videos_exist and labelframes_exist):
    move_to_project_structure()
else:
    print("⏭️ Skipping file movement - dataset already in place")

## 7. Download YOLO Pre-trained Weights

In [ ]:
def download_yolo_weights():
    """Download YOLO pre-trained weights if they don't exist"""
    yolo_weights = {
        'yolov8n.pt': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt',
        'yolo11n.pt': 'https://github.com/ultralytics/assets/releases/download/v8.2.0/yolo11n.pt'
    }
    
    for filename, url in yolo_weights.items():
        filepath = PROJECT_ROOT / filename
        
        if filepath.exists():
            print(f"✅ {filename} already exists")
            continue
        
        print(f"📥 Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"✅ {filename} downloaded successfully")
        except Exception as e:
            print(f"⚠️ Warning: Could not download {filename}: {e}")
            print(f"   You can download it manually from: {url}")

download_yolo_weights()

## 8. Clean Up Temporary Files

In [ ]:
def cleanup_temp_files():
    """Remove temporary download files"""
    if DOWNLOAD_DIR.exists():
        print(f"🧹 Cleaning up temporary files in {DOWNLOAD_DIR}...")
        try:
            shutil.rmtree(DOWNLOAD_DIR)
            print("✅ Temporary files cleaned up successfully")
        except Exception as e:
            print(f"⚠️ Warning: Could not clean up temporary files: {e}")
            print(f"   You can manually delete: {DOWNLOAD_DIR}")

# Only cleanup if we downloaded something
if zip_path:
    cleanup_temp_files()
else:
    print("⏭️ No temporary files to clean up")

## 9. Verify Setup

In [ ]:
def verify_setup():
    """Verify that the dataset setup is complete and correct"""
    print("🔍 Verifying dataset setup...\n")
    
    # Check directories exist
    checks = {
        'data/ directory': DATA_DIR.exists(),
        'data/videos/ directory': VIDEOS_DIR.exists(),
        'data/labelframes/ directory': LABELFRAMES_DIR.exists(),
    }
    
    # Count files
    video_count = len(list(VIDEOS_DIR.glob('*.mp4'))) if VIDEOS_DIR.exists() else 0
    labelframe_count = len(list(LABELFRAMES_DIR.rglob('*.jpg'))) if LABELFRAMES_DIR.exists() else 0
    
    checks.update({
        f'Video files ({video_count} found)': video_count > 0,
        f'Labelframe images ({labelframe_count} found)': labelframe_count > 0,
    })
    
    # Check YOLO weights
    yolo8_exists = (PROJECT_ROOT / 'yolov8n.pt').exists()
    yolo11_exists = (PROJECT_ROOT / 'yolo11n.pt').exists()
    
    checks.update({
        'YOLOv8 weights (yolov8n.pt)': yolo8_exists,
        'YOLO11 weights (yolo11n.pt)': yolo11_exists,
    })
    
    # Print results
    all_good = True
    for check_name, passed in checks.items():
        status = "✅" if passed else "❌"
        print(f"{status} {check_name}")
        if not passed:
            all_good = False
    
    print("\n" + "="*50)
    if all_good:
        print("🎉 SETUP COMPLETE! All files are in place.")
        print("\n📊 Dataset Summary:")
        print(f"   • {video_count:,} video files")
        print(f"   • {labelframe_count:,} labelframe images")
        print(f"   • YOLO weights available")
        print("\n🚀 You can now run the other notebooks in sequence:")
        print("   1. 01_bbox_crops.ipynb")
        print("   2. 02_yolo_oneclass_from_via.ipynb")
        print("   3. 05_vit_behavior_classifier.ipynb")
        print("   4. 06_cow_detection_and_behavior_pipeline.ipynb")
    else:
        print("❌ SETUP INCOMPLETE! Please check the errors above.")
        print("\n🔧 Try running this notebook again or check your Kaggle API setup.")
    
    return all_good

setup_success = verify_setup()

## 🎯 Next Steps

If the setup completed successfully, you're ready to run the main analysis notebooks!

### Recommended execution order:

1. **`01_bbox_crops.ipynb`** - Extract behavior crops from VIA annotations
2. **`02_yolo_oneclass_from_via.ipynb`** - Train YOLO cow detector  
3. **`05_vit_behavior_classifier.ipynb`** - Train ViT behavior classifier
4. **`06_cow_detection_and_behavior_pipeline.ipynb`** - End-to-end pipeline demo

### If you encounter issues:

- **Kaggle API errors**: Check your `kaggle.json` credentials file
- **Download failures**: Try running this notebook again
- **Disk space**: Ensure you have at least 7GB free space
- **Network issues**: The download is large (~6GB), ensure stable internet

---

**Happy analyzing! 🐄📊**